In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Definir la carpeta y armar la lista de archivos

In [2]:
import os

base_dir = "/content/drive/MyDrive/datos DEIS"

years = list(range(2005, 2024))
paths = [os.path.join(base_dir, f"Mortalidad año {y}.xlsx") for y in years]  # ojo: la 'ñ' puede estar como ñ

# Chequeo: cuáles existen y cuáles faltan
missing = [p for p in paths if not os.path.exists(p)]
present = [p for p in paths if os.path.exists(p)]

print("Archivos encontrados:", len(present))
print("Archivos faltantes:", len(missing))
for p in missing[:10]:
    print("Falta:", p)

Archivos encontrados: 19
Archivos faltantes: 0


Leer cada XLSX de forma segura y unir

In [3]:
import pandas as pd

dfs = []

for y in years:
    file_path = os.path.join(base_dir, f"Mortalidad año {y}.xlsx")  # ajustar si era "año"
    df = pd.read_excel(file_path, dtype=str)  # TODO como texto para no deformar nada
    df["anio"] = str(y)
    df["source_file"] = os.path.basename(file_path)  # opcional pero muy útil para auditoría
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True, sort=False)

print("Filas:", combined.shape[0])
print("Columnas:", combined.shape[1])
combined.head(3)

Filas: 922900
Columnas: 8


,PROVRES,SEXO,CAUSA,MAT,GRUPEDAD,CUENTA,anio,source_file
0,50,2,R99,,11_50 a 54,10,2005,Mortalidad año 2005.xlsx
1,14,1,F10,,12_55 a 59,1,2005,Mortalidad año 2005.xlsx
2,62,1,N05,,17_80 y más,1,2005,Mortalidad año 2005.xlsx


## Corroborar que todos las filas fueron cargadas

In [4]:
combined["anio"].value_counts().sort_index()

,count
anio,
2005,45567
2006,45650
2007,47060
2008,46980
2009,47613
2010,47735
2011,47110
2012,47382
2013,48471


In [5]:
cols_by_year = {y: set(dfs[i].columns) for i, y in enumerate(years)}
all_cols = set().union(*cols_by_year.values())

diffs = {y: (all_cols - cols_by_year[y], cols_by_year[y] - all_cols) for y in years}
# Mostrar solo años con diferencias
for y in years:
    missing_cols, extra_cols = diffs[y]
    if missing_cols or extra_cols:
        print(f"Año {y}: faltan {len(missing_cols)} cols, sobran {len(extra_cols)} cols")


## Head general del dataset unificado

In [6]:
combined.head(10)

,PROVRES,SEXO,CAUSA,MAT,GRUPEDAD,CUENTA,anio,source_file
0,50,2,R99,,11_50 a 54,10,2005,Mortalidad año 2005.xlsx
1,14,1,F10,,12_55 a 59,1,2005,Mortalidad año 2005.xlsx
2,62,1,N05,,17_80 y más,1,2005,Mortalidad año 2005.xlsx
3,74,2,C26,,17_80 y más,3,2005,Mortalidad año 2005.xlsx
4,6,1,G30,,13_60 a 64,3,2005,Mortalidad año 2005.xlsx
5,46,2,J96,,15_70 a 74,2,2005,Mortalidad año 2005.xlsx
6,6,1,V49,,13_60 a 64,31,2005,Mortalidad año 2005.xlsx
7,22,2,N17,,17_80 y más,5,2005,Mortalidad año 2005.xlsx
8,26,2,E14,,17_80 y más,8,2005,Mortalidad año 2005.xlsx
9,26,1,J18,,13_60 a 64,3,2005,Mortalidad año 2005.xlsx


In [7]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 922900 entries, 0 to 922899
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   PROVRES      922900 non-null  object
 1   SEXO         922900 non-null  object
 2   CAUSA        922900 non-null  object
 3   MAT          61042 non-null   object
 4   GRUPEDAD     922900 non-null  object
 5   CUENTA       922900 non-null  object
 6   anio         922900 non-null  object
 7   source_file  922900 non-null  object
dtypes: object(8)
memory usage: 56.3+ MB


In [8]:
dfs[0].head(10)   # 2005
dfs[-1].head(10)  # 2023

,PROVRES,SEXO,CAUSA,MAT,GRUPEDAD,CUENTA,anio,source_file
0,14,2,G93,NaN,14_65 a 69,1,2023,Mortalidad año 2023.xlsx
1,14,1,V58,NaN,10_45 a 49,1,2023,Mortalidad año 2023.xlsx
2,50,1,I61,NaN,13_60 a 64,19,2023,Mortalidad año 2023.xlsx
3,14,2,P77,NaN,01_Menor de 1 año,7,2023,Mortalidad año 2023.xlsx
4,14,1,L08,NaN,16_75 a 79,6,2023,Mortalidad año 2023.xlsx
5,30,2,X84,NaN,03_10 a 14,1,2023,Mortalidad año 2023.xlsx
6,6,1,J43,NaN,14_65 a 69,6,2023,Mortalidad año 2023.xlsx
7,6,2,J45,NaN,14_65 a 69,10,2023,Mortalidad año 2023.xlsx
8,6,1,C76,NaN,07_30 a 34,1,2023,Mortalidad año 2023.xlsx
9,2,1,G10,NaN,13_60 a 64,1,2023,Mortalidad año 2023.xlsx


# Exportar dataset en CSV

In [10]:
output_csv = "/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023.csv"
combined.to_csv(output_csv, index=False, encoding="utf-8")
print("Archivos guardados correctamente")

Archivos guardados correctamente


In [11]:
print(os.path.exists(output_csv))

True


# Exportar dataset en Parquet

In [12]:
output_parquet = "/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023.parquet"
combined.to_parquet(output_parquet, index=False)
print("Archivos guardados correctamente")

Archivos guardados correctamente


In [13]:
print(os.path.exists(output_parquet))

True


In [14]:
df = pd.read_parquet("/content/drive/MyDrive/datos DEIS/mortalidad_2005_2023.parquet")

df.shape

(922900, 8)

In [15]:
combined.shape

(922900, 8)